# Lab: Implement a BPE Tokenizer


<a href='https://colab.research.google.com/github/google-deepmind/ai-foundations/blob/master/course_2/gdm_lab_2_4_implement_a_bpe_tokenizer.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg' alt='Open In Colab'/></a>

Implement a subword tokenizer using the byte pair encoding (BPE) algorithm.

25 minutes

## Overview

In this lab, you will implement the **byte pair encoding (BPE) algorithm** that was presented in the previous activity. This will provide you with a  more sophisticated tokenizer than the space tokenizer that you have been using in previous labs. This in return will allow you to train a better model using the Africa Galore dataset.

### What you will learn

By the end of this lab, you will understand:

* The input and output of the byte pair tokenization (BPE) algorithm.
* The iterative steps involved in learning a tokenizer.
* How to implement and run the full BPE algorithm over multiple merge steps.


### Tasks

In this lab, you will:

* Write code to convert each word into a list of characters with `</w>` appended.
* Complete the function `get_pair_frequencies` to count adjacent symbol pairs.
* Fill in logic in `merge_pair_in_word` to replace the most frequent pair.
* Apply merges iteratively to update the corpus and vocabulary.
* Implement the loop in `learn_bpe` to perform a fixed number of merge steps and inspect results.

In [17]:
# Importations
from collections import Counter, defaultdict
import pandas as pd

## Byte pair encoding (BPE)

Recall from the previous article that the BPE algorithm consists of the following steps:

1. **Initialization**: The entire dataset (a text corpus) is split into individual characters and the vocabulary is initialized with the set of unique characters. Spaces are replaced with a special end-of-word symbol (e.g., `</w>`).
2. **Counting**: For each adjacent pair of tokens, count how often each pair appears in your corpus.
3. **Merging**: Choose the adjacent pair of tokens `(p, q)` that appears most frequently in the corpus and add a new merged token `pq` to the vocabulary.
4. **Replacing**: Replace all adjacent pairs of tokens `(p, q)` with the new token `pq` in the corpus.
5. **Repeating**: Repeat steps 2-4 until you reach the target vocabulary size.

### Coding Activity 1: Initialization

You will use the Africa Galore dataset to perform one step of the BPE algorithm before running the algorithm for multiple iterations on a smaller corpus.

Run the following cell to load the dataset.

In [18]:
# Load dataset
url = "https://storage.googleapis.com/dm-educational/assets/ai_foundations/africa_galore.json"

africa_galore = pd.read_json(url)

dataset = africa_galore['description'].values
print(f"Dataset contains {dataset.shape[0]:,} paragraphs")

Dataset contains 232 paragraphs


The initialization step consists of three parts:
1. Converting every space-separated word into a list of characters.
2. Appending the special end-of-word symbol `"</w>"` at the end of each word's list of characters.
3. Adding the set of unique characters to your vocabulary.

<br />

> **💻 Your task:**
>
> Complete the `bpe_initialize` function below such that `corpus` contains a list with each word in the dataset represented as a list of characters and the end-of-word symbol.
>
> Hint: You can turn a string `s` into a list of characters with the `list` function: `list(s)`.

In [19]:
EOW_SYMBOL = "</w>"

# Implements the initialization step of the byte pair encoding algorithm.
def bpe_initialize(dataset: list[str]) -> tuple[list[list[str]], set[str]]:
    corpus = []
    vocabulary = {EOW_SYMBOL}

    for paragraph in dataset:
        for word in paragraph.split(" "):
            # Convert word into chars and add end-of-word symbol ('</w>').
            chars = list(word)
            chars.append(EOW_SYMBOL)
            corpus.append(chars)

            # Append characters to the vocabulary.
            vocabulary.update(chars)

    return corpus, vocabulary

corpus, vocabulary = bpe_initialize(dataset)

# Print the first 10 "words" in the corpus.
print("First 10 words: ")
for word in corpus[:10]:
    print(f" {word}")

print("\n\nVocabulary:")
print(f" {vocabulary}")

First 10 words: 
 ['T', 'h', 'e', '</w>']
 ['L', 'a', 'g', 'o', 's', '</w>']
 ['a', 'i', 'r', '</w>']
 ['w', 'a', 's', '</w>']
 ['t', 'h', 'i', 'c', 'k', '</w>']
 ['w', 'i', 't', 'h', '</w>']
 ['h', 'u', 'm', 'i', 'd', 'i', 't', 'y', ',', '</w>']
 ['b', 'u', 't', '</w>']
 ['t', 'h', 'e', '</w>']
 ['e', 'n', 'e', 'r', 'g', 'y', '</w>']


Vocabulary:
 {'X', 'j', '”', '8', '"', '</w>', '-', 'K', ')', 'm', 'e', 'U', 'x', '9', '4', 'Z', 'é', 'd', 'W', 'Y', 'F', 'c', 'y', 'h', 'M', 'b', 'N', 'q', '—', 's', 'r', 'V', '5', 'p', 'u', ':', 'H', 'L', 'z', '2', '?', 'C', 'w', 'T', 'l', 'A', '7', 'E', '.', 'f', 'a', 'o', 'v', '0', 't', ',', 'I', 'n', 'O', '3', 'k', 'D', 'S', '°', 'B', 'P', 'i', 'R', 'J', '1', 'G', '/', '(', "'", 'g', '6', '“', ';'}



### Coding Activity 2: Counting adjacent token pairs

The second step of the BPE algorithm is to count adjacent token pairs. For every word in the corpus, you have to extract every pair of tokens that appear next to each other and increase their count. For example, when encountering the word represented as `["r", "i", "c", "e", "</w">]`, the counts of the following four pairs should be incremented by 1: `(r,i)`, `(i,c)`, `(c,e)`, `(e, </w>)`.

<br />

> **Your task:**
>
> Complete the `get_pair_frequencies` function below.
>
> This function should return a [Counter](https://docs.python.org/3/library/collections.html#collections.Counter) object whose keys are pairs of tokens and the value is the frequency of the pair in the corpus.
>

In [20]:
# Calculates the frequency of adjacent character pairs in a corpus.
def get_pair_frequencies(corpus: list[list[str]]) -> Counter[tuple[str, str], int]:
    pairs = Counter()
    for word in corpus:
        # Loop through the position of every token but the last one.
        for i in range(len(word) - 1):
            # Create a tuple representing the adjacent pair consisting of the
            # i-th and (i+1)-th token in `word`.
            pair = (word[i], word[i+1]) # My code
            pairs[pair] += 1

    return pairs

pair_freqs = get_pair_frequencies(corpus)
print("The 10 most common pair frequencies are:")
for freq in pair_freqs.most_common(10):
    print(f" {freq}")

The 10 most common pair frequencies are:
 (('e', '</w>'), 2639)
 (('d', '</w>'), 2146)
 (('s', '</w>'), 2078)
 (('a', 'n'), 1883)
 (('t', 'h'), 1869)
 (('i', 'n'), 1822)
 (('h', 'e'), 1735)
 ((',', '</w>'), 1710)
 (('e', 'r'), 1359)
 (('n', 'd'), 1305)


The output shows the most frequent adjacent character pairs and their counts in the corpus. The pairs with `</w>` indicate characters that frequently appear at the end of words. For example, `("e", "</w>")` is the most frequent pair, meaning that many words in the corpus end with the letter "e". Other common pairs like `("i", "n")`, `("a", "n")`, and `("t", "h")` represent frequent character combinations within words. This information is used in the next step of the BPE algorithm to merge the most frequent pairs into new subword tokens.

### Coding Activity 3: Merge the most frequent pair

The third step of the BPE algorithm is to identify the most frequent pair of adjacent tokens and to merge them into a new subword token.

Run the following cell to identify the most frequent token pair and add it to the vocabulary.

In [16]:
most_freq_pair, freq = pair_freqs.most_common(1)[0]

print(f"The most frequent pair is {most_freq_pair} with a frequency of {freq:,}.")

new_token = most_freq_pair[0] + most_freq_pair[1]
vocabulary.add(new_token)

The most frequent pair is ('e', '</w>') with a frequency of 2,639.


> **Your task:**
>
> Complete the `merge_pair_in_word` function below.
>
> This function should replace all instances of `pair_to_merge`
> in `word` with a new subword token that consists of the two
> tokens in `pair_to_merge`.

In [28]:
# Merges adjacent occurrences of a specified pair of characters in a word.
def merge_pair_in_word(word: list[str], pair_to_merge: tuple[str, str]) -> list[str]:
    merged_symbol = pair_to_merge[0] + pair_to_merge[1]
    i = 0
    new_word = []
    while i < len(word):
        # If this position and the next match the pair, merge them.
        if i < len(word) - 1 and word[i] + word[i+1] == merged_symbol:
            new_word.append(merged_symbol)
            i += 2
        else:
            new_word.append(word[i])
            i += 1
    return new_word

print(f"Before merging: {' '.join(corpus[0])}")
new_word = merge_pair_in_word(corpus[0], most_freq_pair)
print(f"After merging:  {' '.join(new_word)}")

Before merging: T h e </w>
After merging:  T h e</w>


### Replace tokens in corpus

Once the new token has been added to the vocabulary, you can use the `merge_pair_in_word` function that you implemented above to replace every occurrence of the most frequent pair with the new merged subword token. This is the fourth step of the BPE algorithm.

Run the following cell to perform the merges in the corpus.

In [29]:
new_corpus = []
for word in corpus:
    new_word = merge_pair_in_word(word, most_freq_pair)
    new_corpus.append(new_word)

# Each iteration of BPE results in a new corpus with merged pairs.
print(
    "The first 10 \"words\" in the corpus, with \"e</w>\" being a single token"
    " now:"
)
for word in new_corpus[:10]:
    print(f"  {word}")

print("\n\nVocabulary:")
print(f"  {vocabulary}")

The first 10 "words" in the corpus, with "e</w>" being a single token now:
  ['T', 'h', 'e</w>']
  ['L', 'a', 'g', 'o', 's', '</w>']
  ['a', 'i', 'r', '</w>']
  ['w', 'a', 's', '</w>']
  ['t', 'h', 'i', 'c', 'k', '</w>']
  ['w', 'i', 't', 'h', '</w>']
  ['h', 'u', 'm', 'i', 'd', 'i', 't', 'y', ',', '</w>']
  ['b', 'u', 't', '</w>']
  ['t', 'h', 'e</w>']
  ['e', 'n', 'e', 'r', 'g', 'y', '</w>']


Vocabulary:
  {'X', 'j', '”', '8', '"', '</w>', '-', 'K', ')', 'm', 'e', 'U', 'x', '9', '4', 'Z', 'é', 'd', 'W', 'Y', 'F', 'c', 'y', 'h', 'M', 'b', 'N', 'q', '—', 's', 'r', 'V', '5', 'p', 'u', ':', 'H', 'L', 'z', '2', '?', 'C', 'w', 'T', 'l', 'A', '7', 'E', '.', 'f', 'a', 'o', 'v', '0', 't', ',', 'I', 'n', 'O', '3', 'k', 'D', 'S', '°', 'B', 'P', 'i', 'R', 'J', '1', 'G', '/', '(', "'", 'g', '6', '“', ';'}


### Repeat

The steps 2, 3, and 4 can now be repeated for either a fixed number of steps or until you reach a target vocabulary size.

On each iteration, exactly one pair of tokens is merged resulting in one more subword token in the vocabulary.

The function below implements the complete BPE algorithm that runs for a fixed number of `num_merges`.

In [31]:
# Learns byte pair encoding (BPE) merge operations from a dataset.
def learn_bpe(dataset: list[str], num_merges: int) -> tuple[list[tuple[str, str]], list[list[str]]]:
    
    # Step 1: Initialize corpus as list of lists of characters + </w>.
    corpus, vocabulary = bpe_initialize(dataset)

    merges = []
    for _ in range(num_merges):
        # Step 2: Count all adjacent-pair frequencies.
        pair_freqs = get_pair_frequencies(corpus)
        if not pair_freqs:
            break

        # Step 3: Pick the most frequent pair.
        most_freq_pair, freq = pair_freqs.most_common(1)[0]
        if freq < 1:
            break  # No more pairs to merge.
        merges.append(most_freq_pair)
        new_token = most_freq_pair[0] + most_freq_pair[1]
        vocabulary.add(new_token)

        # Step 4: Merge that pair in every word of the corpus.
        new_corpus = []
        for word in corpus:
            new_word = merge_pair_in_word(word, most_freq_pair)
            new_corpus.append(new_word)
        corpus = new_corpus

    # Return the list of merges, the vocabulary, and the final tokenized corpus.
    return merges, vocabulary, corpus

### Run BPE on a small corpus

Run the following cell to apply your BPE algorithm to a small collection of words.

In [32]:
sample_text = [
        "desert",
        "deserted",
        "deserts",
        "desert",
        "tested",
        "test",
        "deserted",
        "desert",
        "desertion",
        "desertion",
        "function",
]
# Learn BPE tokenizer with 10 merge operations.
merges, vocabulary, tokenized_corpus = learn_bpe(sample_text, num_merges=10)

print("Learned merges (in order):")
for i, pair in enumerate(merges, start=1):
    print(f" {i}. Merge {pair[0]!r} + {pair[1]!r}  →  {pair[0]+pair[1]!r}")

print("\nFinal tokenized corpus:")
for orig, tokenized in zip(sample_text, tokenized_corpus):
    print(f"  {orig:8s} → {' '.join(tokenized)}")

Learned merges (in order):
 1. Merge 'e' + 's'  →  'es'
 2. Merge 'd' + 'es'  →  'des'
 3. Merge 'des' + 'e'  →  'dese'
 4. Merge 'dese' + 'r'  →  'deser'
 5. Merge 'deser' + 't'  →  'desert'
 6. Merge 'desert' + '</w>'  →  'desert</w>'
 7. Merge 'e' + 'd'  →  'ed'
 8. Merge 'ed' + '</w>'  →  'ed</w>'
 9. Merge 'i' + 'o'  →  'io'
 10. Merge 'io' + 'n'  →  'ion'

Final tokenized corpus:
  desert   → desert</w>
  deserted → desert ed</w>
  deserts  → desert s </w>
  desert   → desert</w>
  tested   → t es t ed</w>
  test     → t es t </w>
  deserted → desert ed</w>
  desert   → desert</w>
  desertion → desert ion </w>
  desertion → desert ion </w>
  function → f u n c t ion </w>


## Summary

In this lab, you implemented the **byte pair encoding (BPE) algorithm**, a cornerstone of modern text tokenization. Through this implementation, you have observed how BPE iteratively **merges** frequent character pairs to create **subword tokens**. This process is crucial for handling the vast and dynamic nature of human language. It is especially important in the context of **large vocabulary sizes** and the need to represent **out-of-vocabulary tokens**. Understanding BPE and other subword tokenization techniques is critical for anyone working with language models. It directly impacts model performance, efficiency, and the ability to handle diverse text data.

In the next module of this course, you will explore how language models encode meaning in a high-dimensional vector space.

---